# TPCRP-CS - cross-cluster diversity modification
Adds a CoreSet-style distance penalty to TPCRP so selected points are spread across the full embedding space, not just locally within clusters

### config

In [ ]:
BUDGETS = [10, 20, 30, 40, 50, 60, 100, 150, 200, 250, 300]

N_REPEATS = 10
CLF_EPOCHS = 200
CLF_LR = 0.025
WEIGHT_DECAY = 5e-4
K_NEIGHBOURS = 20
MAX_CLUSTERS = 500
MIN_CLUSTER_SIZE = 5
SEED = 42

DATA_DIR = "./data" # CIFAR 10 data
EMB_PATH = "../embeddings/embeddings_500epochs.npy"
LBL_PATH = "../labels/labels.npy"


### imports

In [ ]:
import os, time, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
np.random.seed(SEED)

# Machines or Kaggle or Colab
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD = [0.2023, 0.1994, 0.2010]
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

print(f"device : {DEVICE}")




### data

In [ ]:
train_transform = T.Compose(
    [
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)
test_transform = T.Compose(
    [
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)

full_train = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=train_transform
)
test_set = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=test_transform
)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=512, shuffle=False, num_workers=0)

embeddings = np.load(EMB_PATH)
labels = np.load(LBL_PATH)

print("data loaded:", embeddings.shape)


### helpers

In [ ]:
class LabeledSubset(torch.utils.data.Dataset):
    def __init__(self, base, indices):
        self.base, self.indices = base, indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[self.indices[i]]

def train_and_eval(selected_indices, epochs=CLF_EPOCHS, lr=CLF_LR, wd=WEIGHT_DECAY, seed=SEED):
    torch.manual_seed(seed)
    subset = LabeledSubset(full_train, selected_indices)
    loader = torch.utils.data.DataLoader(
        subset, batch_size=min(128, len(subset)), shuffle=True, num_workers=0, drop_last=False
    )

    net = torchvision.models.resnet18(weights=None)
    net.fc = nn.Linear(net.fc.in_features, 10)
    net = net.to(DEVICE)

    optim = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        net.train()
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optim.zero_grad()
            crit(net(imgs), lbls).backward()
            optim.step()
        sched.step()

    net.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            correct += (net(imgs).argmax(1) == lbls).sum().item()
            total += lbls.size(0)
    return 100.0 * correct / total, net



@torch.no_grad()
def get_softmax_all(net):
    plain_transform = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    plain_set = torchvision.datasets.CIFAR10(
        root=DATA_DIR, train=True, download=False, transform=plain_transform
    )
    plain_loader = torch.utils.data.DataLoader(
        plain_set, batch_size=512, shuffle=False, num_workers=0
    )
    net.eval()
    probs = []
    for imgs, _ in plain_loader:
        p = F.softmax(net(imgs.to(DEVICE)), dim=1).cpu().numpy()
        probs.append(p)
    return np.concatenate(probs)

print("Helpers defined.")


### query strategies

In [ ]:
def strategy_random(embeddings, budget, model=None, seed=SEED, **kwargs):
    rng = np.random.default_rng(seed)
    return rng.choice(len(embeddings), budget, replace=False)


def strategy_uncertainty(embeddings, budget, model=None, seed=SEED, **kwargs):
    if model is None:
        return strategy_random(embeddings, budget, seed=seed)
    probs = get_softmax_all(model)
    scores = 1.0 - probs.max(axis=1)
    return np.argsort(-scores)[:budget]


def strategy_margin(embeddings, budget, model=None, seed=SEED, **kwargs):
    if model is None:
        return strategy_random(embeddings, budget, seed=seed)
    probs = get_softmax_all(model)
    sorted_p = np.sort(probs, axis=1)[:, ::-1]
    margin = sorted_p[:, 0] - sorted_p[:, 1]
    return np.argsort(margin)[:budget]


def strategy_entropy(embeddings, budget, model=None, seed=SEED, **kwargs):
    if model is None:
        return strategy_random(embeddings, budget, seed=seed)
    probs = get_softmax_all(model)
    probs = np.clip(probs, 1e-8, 1.0)
    entropy = -(probs * np.log(probs)).sum(axis=1)
    return np.argsort(-entropy)[:budget]



def strategy_coreset(embeddings, budget, model=None, seed=SEED, **kwargs):
    rng = np.random.default_rng(seed)
    N = len(embeddings)
    emb = embeddings.astype(np.float32)

    selected = [int(rng.integers(0, N))]
    min_dist = np.full(N, np.inf)

    for _ in range(budget - 1):
        last = emb[selected[-1]]
        dists = np.linalg.norm(emb - last, axis=1)
        min_dist = np.minimum(min_dist, dists)
        next_idx = int(np.argmax(min_dist))
        selected.append(next_idx)
        min_dist[next_idx] = 0.0

    return np.array(selected)


def strategy_tpcrp(embeddings, budget, model=None, seed=SEED, global_typicality=None, **kwargs):
    K = min(budget, MAX_CLUSTERS)
    if K <= 50:
        km = KMeans(n_clusters=K, random_state=seed, n_init=10)
    else:
        km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init=5, batch_size=2048)
    assignments = km.fit_predict(embeddings)
    sizes = np.bincount(assignments, minlength=K)
    sorted_clusters = np.argsort(-sizes)

    selected = []
    for cid in sorted_clusters:
        if len(selected) >= budget:
            break
        c_idxs = np.where(assignments == cid)[0]
        if len(c_idxs) < MIN_CLUSTER_SIZE:
            continue
        best = c_idxs[np.argmax(global_typicality[c_idxs])]
        selected.append(int(best))
    return np.array(selected)

def strategy_tpcrp_cs(embeddings, budget, model=None, seed=SEED, global_typicality=None, **kwargs):
    K = min(budget, MAX_CLUSTERS)
    if K <= 50:
        km = KMeans(n_clusters=K, random_state=seed, n_init=10)
    else:
        km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init=5, batch_size=2048)
    assignments = km.fit_predict(embeddings)
    sizes = np.bincount(assignments, minlength=K)
    sorted_clusters = np.argsort(-sizes)

    selected = []
    for cid in sorted_clusters:
        if len(selected) >= budget:
            break
        c_idxs = np.where(assignments == cid)[0]
        if len(c_idxs) < MIN_CLUSTER_SIZE:
            continue

        if len(selected) == 0:
            best = c_idxs[np.argmax(global_typicality[c_idxs])]
        else:
            already = embeddings[selected]
            dists = cdist(embeddings[c_idxs], already, metric="euclidean")
            d_min = dists.min(axis=1) #  distance to nearest already selected point
            scores = global_typicality[c_idxs] * d_min
            best = c_idxs[np.argmax(scores)]

        selected.append(int(best))
    return np.array(selected)

STRATEGIES = {"Random":strategy_random,
    "TPCRP":strategy_tpcrp,
    "TPCRP-CS":strategy_tpcrp_cs,}

print("All strategies defined:")
for name in STRATEGIES:
    print(f"  {name}")


### global typicality

In [ ]:
print("computing typicality...")
t0 = time.time()
nbrs = NearestNeighbors(
    n_neighbors=K_NEIGHBOURS + 1, algorithm="auto", metric="euclidean", n_jobs=-1
)

nbrs.fit(embeddings)
dists, _ = nbrs.kneighbors(embeddings)
GLOBAL_TYPICALITY = 1.0 / (dists[:, 1:].mean(axis=1) + 1e-8)

print(f"done in {time.time()-t0:.1f}s")


### sweep

In [ ]:
results = {name: {b:[] for b in BUDGETS} for name in STRATEGIES}

for budget in BUDGETS:
    print(f"budget {budget}")

    for rep in range(N_REPEATS):
        seed_r = SEED + rep * 100
        np.random.seed(seed_r)
        torch.manual_seed(seed_r)

        for name, strategy_fn in STRATEGIES.items():
            idx = strategy_fn(
                embeddings, budget, seed=seed_r, global_typicality=GLOBAL_TYPICALITY
            )
            acc, _ = train_and_eval(idx, seed=seed_r)
            results[name][budget].append(acc)

        accs = "  ".join(f"{n}={results [n ][budget ][-1 ]:.1f}%" for n in STRATEGIES)
        print(f"  rep {rep+1}: {accs}")

print("Done")


### results table

In [ ]:

names = list(STRATEGIES.keys())
print("budget  " + "  ".join(f"{n:>10}" for n in names))
for b in BUDGETS:
    row = f"B={b:<4}   "
    for n in names:
        m = np.mean(results[n][b])
        row += f"  {m:5.2f}%    "
    print(row)


### plots

In [ ]:

BUDGETS_LOW = [10, 20, 30, 40, 50, 60]
BUDGETS_HIGH = [50, 100, 150, 200, 250, 300]

colors = {"Random":"#7F8C8D",
    "TPCRP":"#2980B9",
    "TPCRP-CS":"#E74C3C",}


markers = {"Random":("s", "--"),
    "TPCRP":("o", "-"),
    "TPCRP-CS":("D", "-"),}

means = {n: {b:np.mean(results[n][b]) for b in BUDGETS} for n in STRATEGIES}
stds = {n: {b:np.std(results[n][b]) for b in BUDGETS} for n in STRATEGIES}

def plot_comparison(budget_range, filename, title_suffix):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]

    for n in STRATEGIES:
        m = [means[n][b] for b in budget_range]
        s = [stds[n][b] for b in budget_range]
        mk, ls = markers[n]
        lw = 2.4 if n != "Random" else 1.4
        ax.errorbar(
            budget_range,
            m,
            yerr=s,
            fmt=f"{mk}{ls}",
            color=colors[n],
            linewidth=lw,
            capsize=3,
            markersize=6,
            label=n,
            zorder=10 if n == "TPCRP-CS" else 3,
        )


    ax.set_xlabel("Budget B (total labeled examples)", fontsize=11)
    ax.set_ylabel("Test accuracy (%)", fontsize=11)
    ax.set_title(f"TPCRP vs TPCRP-CS — CIFAR-10\n{title_suffix}", fontsize=11)
    ax.set_xticks(budget_range)
    ax.set_xticklabels([f"{b}\n({b//10}/cls)" for b in budget_range], fontsize=8.5)
    ax.legend(fontsize=10, framealpha=0.9)
    ax.grid(axis="y", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax2 = axes[1]
    rand = np.array([means["Random"][b] for b in budget_range])
    for n in ["TPCRP", "TPCRP-CS"]:
        gains = np.array([means[n][b] for b in budget_range]) - rand
        mk, ls = markers[n]
        ax2.plot(
            budget_range,
            gains,
            f"{mk}{ls}",
            color=colors[n],
            linewidth=2.4,
            markersize=6,
            label=n,
        )


    ax2.axhline(0, color="black", linewidth=1.0)
    ax2.set_xlabel("Budget B", fontsize=11)
    ax2.set_ylabel("Gain over Random (%)", fontsize=11)
    ax2.set_title("Accuracy Gain vs Random", fontsize=11)
    ax2.set_xticks(budget_range)
    ax2.legend(fontsize=10)
    ax2.grid(axis="y", alpha=0.25)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(filename, dpi = 150, bbox_inches = "tight")
    plt.show()
    print(f"Saved:  {filename}")

plot_comparison(BUDGETS_LOW, "./modification_low_budget.png",  "Low budget [10-60]")
plot_comparison(BUDGETS_HIGH, "./modification_high_budget.png", "Higher budget [50-300]")


### summary

In [ ]:
names = list(STRATEGIES.keys())
print("average accuracy per strategy:")

for n in names:
    avg = np.mean([np.mean(results[n][b]) for b in BUDGETS])
    print(f"  {n}: {avg:.2f}%")

tpcrp_avg = np.mean([np.mean(results["TPCRP"][b]) for b in BUDGETS])
cs_avg = np.mean([np.mean(results["TPCRP-CS"][b]) for b in BUDGETS])


print(f"CS gain over TPCRP: {cs_avg - tpcrp_avg:+.2f}%")
